# 파트 예측: 국내 주식 섹터 상승 예측 프로젝트

이 노트북은 GitHub 포트폴리오 발표용으로, 프로젝트의 전체 흐름을 한 번에 설명하고 재현하기 위한 실행형 문서입니다.

목표는 개별 종목을 바로 맞히는 것이 아니라, 다음 거래일에 상대적으로 강할 가능성이 높은 국내 주식 섹터를 먼저 예측하는 것입니다. 이후 섹터 단위 판단을 기반으로 개별 종목 후보를 좁혀가는 구조입니다.

핵심 질문은 다음과 같습니다.

- 뉴스, 검색 관심도, FOMO 신호, 환율, 수급, 장중 차트 흐름은 다음 거래일 섹터 움직임을 설명할 수 있는가?
- 단순 상승 여부뿐 아니라 시장 대비 강도, 거래 가능한 상승 가능성, 과열 위험까지 함께 판단할 수 있는가?
- 모델이 너무 자신 있게 예측하는 구간을 확률 보정으로 낮춰 실전 의사결정에 더 안전하게 쓸 수 있는가?


## 전체 파이프라인

```mermaid
flowchart TD
    A[API 및 공개 데이터 수집] --> B[원천 데이터 저장]
    B --> C[전처리와 결측치 처리]
    C --> D[섹터 단위 파생변수 생성]
    D --> E[상승 여부 및 기대수익률 모델 학습]
    E --> F[랭킹 모델 V3, V4, V5 학습]
    F --> G[확률 보정 및 리스크 게이트]
    G --> H[내일 섹터 예측표 생성]
    H --> I[예측 검증 및 일일 기록]
```

프로젝트는 단일 모델 하나로 끝나는 구조가 아니라, 데이터 수집, 전처리, 모델 학습, 보정, 예측 검증이 매일 반복되는 운영형 머신러닝 프로젝트입니다.


## 1. 실행 환경 준비

API 키는 코드에 직접 쓰지 않고 프로젝트 루트의 `.env` 파일에서 읽습니다. GitHub에 올릴 때는 실제 키가 들어간 `.env`를 올리지 않고, `.env.example`만 공개합니다.


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPTS = ROOT / "scripts"
DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
REPORTS = ROOT / "reports"

sys.path.insert(0, str(SCRIPTS))

print("project root:", ROOT)
print("scripts:", SCRIPTS.exists())
print("processed data:", PROCESSED.exists())
print("reports:", REPORTS.exists())


In [ ]:
def load_dotenv(path: Path) -> dict:
    loaded = {}
    if not path.exists():
        print(".env 파일이 없어 API 기반 수집은 건너뜁니다:", path)
        return loaded
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)
        loaded[key] = bool(value)
    return loaded

loaded_env = load_dotenv(ROOT / ".env")
safe_keys = [
    "NAVER_CLIENT_ID", "NAVER_CLIENT_SECRET", "NAVER_DATALAB_CLIENT_ID", "NAVER_DATALAB_CLIENT_SECRET",
    "ECOS_API_KEY", "KRX_API_KEY", "DART_API_KEY", "KIS_APP_KEY", "KIS_APP_SECRET", "KIS_IS_MOCK"
]
pd.DataFrame({"env_key": safe_keys, "loaded": [bool(os.environ.get(k, "").strip()) for k in safe_keys]})


## 2. 데이터 수집 단계

수집 데이터는 크게 여섯 묶음입니다.

| 데이터 | 수집 스크립트 | 의미 |
|---|---|---|
| ECOS 환율 | `collect_fx_data.py` | 원/달러 등 거시 변수와 환율 충격 |
| NAVER 뉴스 | `collect_naver_news_search.py` | 섹터별 뉴스 관심도와 FOMO 신호 |
| NAVER/KIS 수급 | `collect_naver_investor_flows.py`, `collect_kis_investor_flows.py` | 개인, 외국인, 기관 수급 흐름 |
| pykrx/KRX | `collect_pykrx_data.py`, `collect_krx_openapi.py` | 일봉, 거래대금, 시가총액, 공식 시장 데이터 |
| OpenDART | `collect_dart_disclosures.py` | 공시 이벤트와 섹터별 공시 강도 |
| KIS/Naver 실시간 | `collect_kis_intraday.py`, `collect_naver_realtime.py` | 장중 섹터 차트 흐름과 반등 신호 |

아래 셀은 실제 수집을 실행할 수 있지만, API 호출과 시간이 필요하기 때문에 기본값은 `False`입니다. 발표 때는 코드 구조와 산출 파일을 보여주고, 필요할 때만 `RUN_COLLECTION = True`로 바꿔 실행하면 됩니다.


In [ ]:
RUN_COLLECTION = False

def run_script(script_name: str, required_env: tuple[str, ...] = (), run: bool = False) -> None:
    missing = [key for key in required_env if not os.environ.get(key, "").strip()]
    script_path = SCRIPTS / script_name
    if missing:
        print(f"SKIP {script_name}: missing env {missing}")
        return
    if not script_path.exists():
        print(f"SKIP {script_name}: file not found")
        return
    if not run:
        print(f"READY {script_name}")
        return
    print(f"RUN {script_name}")
    subprocess.run([sys.executable, str(script_path)], cwd=ROOT, check=True)

collection_steps = [
    ("collect_fx_data.py", ("ECOS_API_KEY",)),
    ("collect_naver_news_search.py", ("NAVER_CLIENT_ID", "NAVER_CLIENT_SECRET")),
    ("collect_naver_investor_flows.py", ()),
    ("collect_kis_investor_flows.py", ("KIS_APP_KEY", "KIS_APP_SECRET")),
    ("collect_pykrx_data.py", ()),
    ("collect_dart_disclosures.py", ("DART_API_KEY",)),
    ("collect_krx_openapi.py", ("KRX_API_KEY",)),
    ("collect_kis_intraday.py", ("KIS_APP_KEY", "KIS_APP_SECRET")),
    ("collect_naver_realtime.py", ()),
    ("collect_kaggle_global_market_data.py", ()),
]

for script_name, required in collection_steps:
    run_script(script_name, required, run=RUN_COLLECTION)


## 3. 원천 데이터와 가공 데이터 확인

수집된 데이터는 `data/raw`에 원천 형태로 저장하고, 모델이 쓰기 좋은 형태는 `data/processed`에 저장합니다. 이 구조는 GitHub 포트폴리오에서 재현성과 관리성을 보여주기 좋습니다.


In [ ]:
def file_inventory(folder: Path, pattern: str = "*.csv") -> pd.DataFrame:
    rows = []
    for path in sorted(folder.rglob(pattern)) if folder.exists() else []:
        rows.append({
            "file": str(path.relative_to(ROOT)),
            "size_mb": round(path.stat().st_size / 1024 / 1024, 3),
            "modified": pd.Timestamp(path.stat().st_mtime, unit="s"),
        })
    return pd.DataFrame(rows)

raw_files = file_inventory(RAW)
processed_files = file_inventory(PROCESSED)
display(raw_files.tail(10))
display(processed_files.tail(10))


## 4. 전처리와 특성 선택

전처리 단계에서는 모델 성능을 해칠 수 있는 값을 정리합니다.

- `inf`, `-inf` 값을 결측치로 변환
- 결측률이 너무 높은 변수 제거
- `next_`, `future_`, `target_`, `actual_`처럼 미래 정보를 포함할 가능성이 있는 누수 변수 제거
- 수급 원자료처럼 스케일이 큰 값은 signed log, 일자별 rank로 변환
- 중간 정도 결측률을 가진 변수에는 missing flag 추가
- 이상치는 winsorizing으로 완화


In [ ]:
from preprocess_features import prepare_model_features, select_numeric_features, feature_selection_report

sample_path = REPORTS / "sector_model_v2_panel.csv"
if sample_path.exists():
    panel_sample = pd.read_csv(sample_path, encoding="utf-8-sig")
    prepared_sample = prepare_model_features(panel_sample)
    excluded = {"target_next_return", "target_up", "target_excess_return"}
    selected_features = select_numeric_features(prepared_sample, excluded=excluded)
    report = feature_selection_report(prepared_sample, selected_features, excluded=excluded)
    print("panel shape:", panel_sample.shape)
    print("prepared shape:", prepared_sample.shape)
    print("selected numeric features:", len(selected_features))
    display(pd.Series(report).to_frame("value"))
else:
    print("모델 패널 파일이 아직 없습니다. train_sector_return_model_v2.py를 먼저 실행하세요.")


## 5. 섹터 단위 변수 생성

이 프로젝트는 종목 하나하나를 바로 예측하기 전에 섹터 단위로 시장을 봅니다. 이유는 개별 종목보다 섹터가 노이즈가 적고, 뉴스와 수급이 집단적으로 반응하는 경우가 많기 때문입니다.

대표 파생변수는 다음과 같습니다.

- 시장 방향성: 전체 섹터 평균 수익률, 상승 섹터 비율, 하락 섹터 비율
- 상대 강도: 해당 섹터 수익률과 시장 평균 수익률의 차이
- FOMO/관심도: 뉴스 건수, 검색 관심도, 관심도 가속, 과열 위험
- 수급: 개인, 외국인, 기관 순매수 흐름과 이동평균, z-score
- 거래대금: 섹터 전체 거래대금, 상위 대형주 집중도
- 환율 민감도: 환율 변화와 섹터 반응의 관계
- 캘린더 제어: 휴장일, 주말 갭, 다음 거래일 여부


In [ ]:
RUN_FEATURE_BUILD = False

if RUN_FEATURE_BUILD:
    from train_sector_return_model_v2 import build_panel, numeric_features
    model_panel = build_panel()
else:
    panel_path = REPORTS / "sector_model_v2_panel.csv"
    model_panel = pd.read_csv(panel_path, encoding="utf-8-sig") if panel_path.exists() else pd.DataFrame()

if not model_panel.empty:
    print("date range:", model_panel["date"].min(), "~", model_panel["date"].max())
    print("rows:", len(model_panel), "sectors:", model_panel["sector"].nunique())
    key_cols = [
        "date", "sector", "target_next_return", "target_up", "target_excess_return",
        "market_direction_score", "sector_vs_market_return_gap", "news_3d", "live_fomo_score"
    ]
    display(model_panel[[c for c in key_cols if c in model_panel.columns]].tail(12))
else:
    print("모델 패널을 찾지 못했습니다.")


## 6. 모델 구성

현재 모델은 딥러닝이 아니라 머신러닝 앙상블 중심입니다.

### V2: 다음 거래일 섹터 수익률과 상승 확률
- Ridge, Gradient Boosting, Extra Trees 회귀 모델로 기대수익률 예측
- Logistic Regression, Random Forest 분류 모델로 상승 확률 예측
- 보조 라벨: 의미 있는 상승, 시장 대비 초과 상승, 거래 가능한 상승

### V3~V5: 섹터 랭킹 모델
- 단순 상승 여부보다 `내일 어떤 섹터가 상대적으로 강할까`를 순위로 판단
- V4는 시장 국면과 리스크 게이트를 추가
- V5는 글로벌 시장 변수, FOMO 과열, 관심도 반전, 거래대금 집중도를 추가

### 확률 보정
- 모델이 산출한 확률이 실제 적중률보다 과하게 높을 수 있어 백테스트 구간에서 확률 구간별 실제 적중률을 계산
- 보정 확률을 `calibrated_*_proba` 컬럼으로 저장
- 최종 의사결정은 원래 확률보다 보정 확률을 더 신뢰


In [ ]:
RUN_TRAINING = False

training_steps = [
    "train_sector_return_model_v2.py",
    "train_sector_rank_model_v3.py",
    "train_sector_rank_model_v4.py",
    "train_sector_rank_model_v5.py",
]

for script_name in training_steps:
    run_script(script_name, run=RUN_TRAINING)


## 7. 모델 성능과 확률 보정 결과 확인

GitHub 포트폴리오에서는 정확도가 매우 높다고 말하는 것보다, 모델이 어디서 약하고 어떤 방식으로 보완했는지를 보여주는 것이 더 설득력 있습니다. 이 프로젝트는 확률 보정 리포트를 통해 과신 여부를 점검합니다.


In [ ]:
def show_markdown_head(path: Path, n: int = 80) -> None:
    if not path.exists():
        print("missing:", path)
        return
    lines = path.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[:n]))

show_markdown_head(REPORTS / "sector_model_v2_report.md", 60)
print("\n" + "=" * 80 + "\n")
show_markdown_head(REPORTS / "probability_calibration_report.md", 40)


## 8. 내일 예측 생성

최종 산출물은 `reports/tomorrow_sector_prediction.csv`입니다.

여기에는 기존 모델 점수와 새 의사결정 레이어가 함께 들어갑니다.

- `tomorrow_action`: 기존 모델 기준의 후보 판단
- `decision_action`: 확률 보정과 리스크 제어를 반영한 보수적 판단
- `decision_confidence_label`: 예측 신뢰도
- `calibrated_*_proba`: 백테스트 기반 보정 확률


In [ ]:
RUN_PREDICTION = False

prediction_steps = [
    "build_advisor_report.py",
    "build_intraday_rebound_signals.py",
    "compute_weekend_signal_effects.py",
    "build_tomorrow_prediction.py",
    "evaluate_prediction_accuracy.py",
]

for script_name in prediction_steps:
    run_script(script_name, run=RUN_PREDICTION)


In [ ]:
prediction_path = REPORTS / "tomorrow_sector_prediction.csv"
summary_path = REPORTS / "tomorrow_sector_prediction_summary.json"

if prediction_path.exists():
    pred = pd.read_csv(prediction_path, encoding="utf-8-sig")
    display_cols = [
        "sector", "tomorrow_action", "decision_action", "decision_score", "decision_confidence_label",
        "ensemble_v2_next_return_pred", "quality_adjusted_up_proba", "calibrated_quality_adjusted_up_proba",
        "excess_up_proba", "calibrated_excess_up_proba", "calibration_adjusted_grade"
    ]
    display(pred[[c for c in display_cols if c in pred.columns]].head(12))
else:
    print("missing prediction file:", prediction_path)

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    display(pd.Series(summary).to_frame("value"))


## 9. 포트폴리오 관점에서 이 프로젝트의 의미

이 프로젝트는 단순히 주식 예측 결과를 하나 만드는 것이 아니라, 실제 데이터 프로젝트의 전 과정을 보여줍니다.

1. 여러 API와 공개 데이터를 연결했다.
2. 원천 데이터와 가공 데이터를 분리했다.
3. 결측치, 이상치, 데이터 누수 가능성을 전처리 단계에서 관리했다.
4. 뉴스, 수급, 환율, 거래대금, 시장 국면을 섹터 단위 파생변수로 만들었다.
5. 상승 여부, 기대수익률, 섹터 랭킹을 서로 다른 모델 관점으로 예측했다.
6. 모델 확률이 과신되는 문제를 백테스트 기반 확률 보정으로 개선했다.
7. 매일 예측하고, 다음 날 실제 결과와 비교해 기록하는 운영 루프를 만들었다.

따라서 이 프로젝트는 GitHub 포트폴리오에서 데이터 수집, 머신러닝 모델링, 자동화, 결과 기록까지 연결한 개인 프로젝트로 설명할 수 있습니다.


## 10. 발표용 한 문장 요약

> 이 프로젝트는 뉴스, 검색 관심도, 환율, 수급, 거래대금, 장중 차트 데이터를 통합해 다음 거래일 국내 주식 섹터의 상승 가능성과 상대 강도를 예측하고, 예측 확률의 과신을 보정해 실전 의사결정에 사용할 수 있는 섹터 후보를 생성하는 머신러닝 기반 포트폴리오 프로젝트입니다.
